# Mushroom Classification Project

In diesem Notebook arbeiten wir mit dem Mushroom Classification Datensatz. Ziel ist es vorherzusagen, ob ein Pilz essbar (`e`) oder giftig (`p`) ist.

Wir haben versucht, den Ablauf möglichst klar zu halten: Daten laden, kurz anschauen, einfache Datenprobleme beheben, ein Klassifikationsmodell trainieren und die Ergebnisse bewerten.

## 1. Imports

Wir benutzen vor allem `pandas` für die Daten und `scikit-learn` für das Modell.

In [ ]:
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 30)

## 2. Daten laden

Die Datei sieht aus wie eine CSV-Datei, ist aber nicht mit Komma getrennt. Der Trenner ist `~`.

Beim ersten Einlesen gab es außerdem ein paar Zeilen mit zu vielen leeren Feldern am Ende. Deshalb laden wir die Datei etwas vorsichtiger und schneiden nur leere Zusatzfelder am Zeilenende weg.

In [ ]:
raw_path = Path("../data/raw/mushroom_classification_data.csv")

expected_columns = 23
cleaned_lines = []
fixed_lines = []

for line_number, line in enumerate(raw_path.read_text(encoding="utf-8").splitlines(), start=1):
    parts = line.split("~")

    if len(parts) > expected_columns and all(value == "" for value in parts[expected_columns:]):
        fixed_lines.append(line_number)
        parts = parts[:expected_columns]

    cleaned_lines.append("~".join(parts))

df = pd.read_csv(StringIO("\n".join(cleaned_lines)), sep="~")

print("Geladene Daten:", df.shape)
print("Korrigierte Zeilen mit leeren Zusatzfeldern:", fixed_lines)
df.head()

## 3. Erster Überblick

Jetzt schauen wir uns an, wie groß der Datensatz ist, wie die Zielvariable verteilt ist und ob es auffällige Werte gibt.

In [ ]:
print("Zeilen und Spalten:", df.shape)
print("Doppelte Zeilen:", df.duplicated().sum())

print("\nVerteilung der Zielvariable:")
print(df["poison"].value_counts())

print("\nProzent:")
print((df["poison"].value_counts(normalize=True) * 100).round(1))

In [ ]:
question_marks = (df == "?").sum()
question_marks[question_marks > 0]

In [ ]:
df.nunique().sort_values()

Erste Beobachtungen:

- Der Datensatz hat 8.124 Zeilen.
- Die Klassen sind ziemlich ausgeglichen: etwas mehr essbare als giftige Pilze.
- `stalk-root` enthält viele `?`-Werte.
- `veil-type` hat nur einen einzigen Wert und ist für das Modell deshalb nicht hilfreich.

## 4. Einfache Bereinigung

Beim Durchschauen sind ein paar offensichtlich komische Werte aufgefallen, zum Beispiel `d**` oder Zahlen in Spalten, die eigentlich Buchstaben-Kategorien enthalten sollten.

Wir ersetzen diese Werte durch `?`, also durch "unbekannt". Außerdem speichern wir eine bereinigte Version der Daten.

In [ ]:
weird_values = ["d**", "0.34", "0.3", "0.4"]

df_clean = df.replace(weird_values, "?")
df_clean.to_csv("../data/processed/mushroom_classification_clean.csv", index=False)

print("Bereinigte Datei gespeichert.")
print((df_clean == "?").sum()[(df_clean == "?").sum() > 0])

## 5. Einige Zusammenhänge anschauen

Da alle Merkmale kategorial sind, sind Kreuztabellen hier nützlich. Besonders `odor` scheint stark mit der Zielvariable zusammenzuhängen.

In [ ]:
pd.crosstab(df_clean["odor"], df_clean["poison"])

In [ ]:
pd.crosstab(df_clean["spore-print-color"], df_clean["poison"])

## 6. Machine Learning vorbereiten

Wir machen daraus ein überwachtes Klassifikationsproblem:

- Zielvariable: `poison`
- Eingaben: alle anderen Spalten außer `veil-type`
- `?` behandeln wir als fehlende Werte
- kategoriale Variablen werden mit One-Hot-Encoding umgewandelt

Als Modell nutzen wir einen Decision Tree, weil er für diesen Datensatz gut passt und leichter erklärbar ist als viele andere Modelle.

In [ ]:
y = df_clean["poison"].map({"e": 0, "p": 1})
X = df_clean.drop(columns=["poison", "veil-type"]).replace("?", np.nan)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

categorical_columns = X.columns.tolist()

preprocessing = ColumnTransformer(
    transformers=[
        (
            "cat",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            categorical_columns,
        )
    ]
)

## 7. Baseline

Als Vergleich nehmen wir zuerst ein Dummy-Modell. Es lernt nichts Sinnvolles, sondern sagt einfach immer die häufigere Klasse voraus.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
print("Baseline Accuracy:", round(accuracy_score(y_test, baseline_pred), 4))

## 8. Decision Tree Modell

Jetzt trainieren wir das eigentliche Modell.

In [ ]:
model = Pipeline(
    steps=[
        ("preprocessing", preprocessing),
        ("decision_tree", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Decision Tree Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["edible", "poisonous"]))

## 9. Ergebnis

Das Decision-Tree-Modell ist sehr viel besser als die Baseline. Das liegt wahrscheinlich daran, dass einige Merkmale im Datensatz sehr stark mit `poison` zusammenhängen, zum Beispiel `odor`.

Besonders wichtig ist hier der Recall für die giftige Klasse. Wenn ein giftiger Pilz fälschlicherweise als essbar vorhergesagt wird, wäre das der gefährlichste Fehler.

## 10. Fazit

Der Datensatz eignet sich gut für ein Klassifikationsmodell. Nach einer einfachen Bereinigung konnten wir ein Decision-Tree-Modell trainieren, das sehr gute Ergebnisse liefert.

Für eine echte Anwendung müsste man natürlich vorsichtig sein, weil Pilze ein sicherheitskritisches Thema sind. Für dieses Kursprojekt zeigt das Modell aber gut, wie man kategoriale Daten vorbereitet und ein Klassifikationsmodell mit scikit-learn trainiert.

## 11. LLM-Nutzung

Wir haben ein LLM zur Unterstützung benutzt, vor allem um die Aufgabenstellung zu strukturieren, Datenprobleme zu erkennen und Formulierungen für das Notebook zu verbessern. Die Schritte wurden von uns überprüft und sollen von allen Gruppenmitgliedern erklärt werden können.